# Building a Retriever
Load - Split - Retriever



In [ ]:
%pip install -U langchain langchain-ollama langchain-chroma langchain-community pypdf --quiet

Import libraries

In [3]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

Initializing the models: main LLM and embeddings model

In [4]:
llm = ChatOllama(model="gemma4:e4b", temperature=0)
embeddings = OllamaEmbeddings(model="embeddinggemma")

Ingest files from the data directory

In [6]:
loader = PyPDFDirectoryLoader("./data")
docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loaded 5 documents


Let's examine the documents

In [7]:
from pprint import pprint
pprint(docs[0].metadata)
print(docs[0].page_content[:300], " ...")


{'author': '',
 'creationdate': '2026-05-02T14:01:34+10:00',
 'creator': 'PyPDF',
 'moddate': '2026-05-02T14:01:34+10:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Microsoft: Print To PDF',
 'source': 'data\\green_house_effect.pdf',
 'title': 'green_house_effect.txt - Notepad',
 'total_pages': 2}
## The Earth’s Blanket: Understanding the Greenhouse Effect and Its Perils
The Earth’s climate system is regulated by a delicate energy balance, and the 
mechanism responsible for keeping our planet within the narrow habitable range is 
the natural greenhouse effect. Far from being a purely negative  ...


Split the do documents into smaller chunks. 1000 chars per chunk is a good practice

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 14 chunks


Let's examine the chunks

In [ ]:
pprint(chunks[0].metadata)
print(chunks[0].page_content[:300], " ...")

{'author': '',
 'creationdate': '2026-05-02T14:01:34+10:00',
 'creator': 'PyPDF',
 'moddate': '2026-05-02T14:01:34+10:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Microsoft: Print To PDF',
 'source': 'data\\green_house_effect.pdf',
 'title': 'green_house_effect.txt - Notepad',
 'total_pages': 2}
## The Earth’s Blanket: Understanding the Greenhouse Effect and Its Perils
The Earth’s climate system is regulated by a delicate energy balance, and the 
mechanism responsible for keeping our planet within the narrow habitable range is 
the natural greenhouse effect. Far from being a purely negative  ...


Initialize ChromaDB with local storage for persistence and create a retriever

In [10]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
#Retrieve 3 chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

Testing the retriever

In [11]:
retrieved = retriever.invoke("photosynthesis")
print(f"We retrieved {len(retrieved)} chunks")
pprint(retrieved[0].metadata)
print(retrieved[0].page_content[:300], " ...")

We retrieved 3 chunks
{'author': '',
 'creationdate': '2026-05-02T13:57:26+10:00',
 'creator': 'PyPDF',
 'moddate': '2026-05-02T13:57:26+10:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Microsoft: Print To PDF',
 'source': 'data\\photosynthesis - Copy.pdf',
 'title': 'photosynthesis.txt - Notepad',
 'total_pages': 1}
The Engine of Life: How Photosynthesis Works
Photosynthesis is arguably the single most important biological process on Earth. It
is the chemical mechanism by which plants, algae, and certain bacteria convert the 
energy from sunlight into stored chemical energy, providing the fundamental energy 
so  ...


# Need to keep playing with this
Can I also retrieve the score?

In [ ]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={
        "k": 5,  # Return the top 5 most relevant documents
        "score_threshold": 0.7  # Only return documents with a similarity score of 0.7 or higher
    }
)